# California Multi-hazard 2024 Exploration

Explore four California event sources together beginning in 2024:

- Eagle-I county outages
- CPUC PSPS events
- CPUC wildfire ignition events
- Participant interview event references

The notebook builds two outputs:

- an interactive Plotly timeline with hover details
- a map-based movie of California counties and transmission lines with event overlays where spatial information exists

In [ ]:
import io
import json
import re
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import imageio.v3 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)

EAGLEI_CSV = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I/eaglei_outages_2024.csv"
PSPS_CSV = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/CPUC_CA_2024_PSPSevents.csv"
IGNITION_XLSX = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/CA_events_data/Ignition Events 2015-2025_CPUC.xlsx"
PARTICIPANT_XLSX = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/CA_events_data/Participant Events - Cathcart_May2026.xlsx"
TRANSMISSION_GEOJSON = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/U.S._Electric_Power_Transmission_Lines.geojson"
COUNTY_URL = "https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip"
ANALYSIS_CRS = "EPSG:4326"
PROJECTED_CRS = "EPSG:5070"
SOURCE_COLORS = {
    "EAGLE-I": "royalblue",
    "PSPS": "darkorange",
    "CPUC Ignition": "firebrick",
    "Interview reference": "purple",
}

CITY_HINTS = {
    "bishop": "Inyo",
    "placerville": "El Dorado",
    "temecula": "Riverside",
    "colusa": "Colusa",
    "ventura": "Ventura",
}

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).replace("\u2013", "-").replace("\u2014", "-").replace("\u2018", "'").replace("\u2019", "'").strip()

def parse_any_datetime(value):
    text = clean_text(value)
    if not text:
        return pd.NaT
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        match = re.search(r"(20\d{2})", text)
        if match:
            parsed = pd.to_datetime(f"{match.group(1)}-01-01", errors="coerce")
    return parsed

def load_ca_counties(cache_dir="./_cache"):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    zip_path = cache_dir / "tl_2024_us_county.zip"
    extract_dir = cache_dir / "tl_2024_us_county"
    shp_path = extract_dir / "tl_2024_us_county.shp"

    if not shp_path.exists():
        if not zip_path.exists():
            response = requests.get(COUNTY_URL, timeout=120)
            response.raise_for_status()
            zip_path.write_bytes(response.content)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

    counties = gpd.read_file(shp_path).to_crs(ANALYSIS_CRS)
    counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)
    ca = counties[counties["STATEFP"] == "06"].copy().reset_index(drop=True)
    ca["county_name"] = ca["NAME"].astype(str)
    return ca

def load_transmission_lines(path=TRANSMISSION_GEOJSON):
    lines = gpd.read_file(path)
    if lines.crs is None or lines.crs.to_string() != ANALYSIS_CRS:
        lines = lines.to_crs(ANALYSIS_CRS)
    return lines

def county_name_from_text(text, county_names):
    raw = clean_text(text).lower()
    if not raw:
        return None
    for hint, county in CITY_HINTS.items():
        if hint in raw:
            return county
    for county in sorted(county_names, key=len, reverse=True):
        if re.search(rf"\b{re.escape(county.lower())}\b", raw):
            return county
    return None

def active_days(start_time, end_time):
    start = pd.to_datetime(start_time).normalize()
    end = pd.to_datetime(end_time).normalize()
    if pd.isna(start) or pd.isna(end) or end < start:
        return pd.DatetimeIndex([])
    return pd.date_range(start, end, freq="D")

def geo_to_rgba(fig):
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())

In [ ]:
ca_counties = load_ca_counties()
ca_counties = ca_counties[["GEOID", "NAME", "county_name", "geometry"]].copy()
ca_union = ca_counties.geometry.unary_union
ca_centroid = ca_union.centroid
county_centroids = ca_counties.copy()
county_centroids["centroid"] = county_centroids.geometry.centroid
county_centroid_lookup = dict(zip(county_centroids["county_name"].str.lower(), county_centroids["centroid"]))
county_geom_lookup = dict(zip(ca_counties["county_name"].str.lower(), ca_counties.geometry))

# -------------------------------------------------------------------
# Eagle-I county outages
# -------------------------------------------------------------------
eaglei = pd.read_csv(EAGLEI_CSV)
eaglei["start_time"] = pd.to_datetime(eaglei["run_start_time"], errors="coerce")
eaglei = eaglei.dropna(subset=["start_time"]).copy()
eaglei = eaglei[eaglei["state"].astype(str).str.lower().eq("california")].copy()
eaglei["fips_code"] = eaglei["fips_code"].astype(str).str.zfill(5)
eaglei["end_time"] = eaglei["start_time"] + pd.Timedelta(hours=6)
eaglei["event_date"] = eaglei["start_time"].dt.floor("D")
eaglei_agg = (
    eaglei.groupby(["event_date", "fips_code", "county"], as_index=False)
          .agg(customers_out=("customers_out", "max"), total_customers=("total_customers", "max"), n_rows=("customers_out", "size"))
)
eaglei_agg["source"] = "EAGLE-I"
eaglei_agg["event_type"] = "County outage"
eaglei_agg["event_name"] = eaglei_agg["county"].astype(str) + " county outage"
eaglei_agg["start_time"] = eaglei_agg["event_date"]
eaglei_agg["end_time"] = eaglei_agg["event_date"] + pd.Timedelta(hours=6)
eaglei_agg["location_text"] = eaglei_agg["county"].astype(str) + ", California"
eaglei_agg["detail_text"] = "Peak county outage observations from Eagle-I."
eaglei_agg["event_value"] = pd.to_numeric(eaglei_agg["customers_out"], errors="coerce")
eaglei_agg["county_name"] = eaglei_agg["county"].astype(str)
eaglei_agg["plot_end_time"] = eaglei_agg["end_time"]

# -------------------------------------------------------------------
# PSPS events
# -------------------------------------------------------------------
psps = pd.read_csv(PSPS_CSV)
for col in ["first_date_of_poc", "de_energization_starting_date", "full_restoration_date", "CreationDate", "EditDate"]:
    if col in psps.columns:
        psps[col] = pd.to_datetime(psps[col], errors="coerce")
psps["start_time"] = psps["de_energization_starting_date"].fillna(psps["first_date_of_poc"])
psps["end_time"] = psps["full_restoration_date"].fillna(psps["start_time"] + pd.Timedelta(hours=12))
psps = psps.dropna(subset=["start_time"]).copy()
psps = psps[(psps["start_time"].dt.year >= 2024) & (psps["start_time"].dt.year <= 2024)].copy()
psps["source"] = "PSPS"
psps["event_type"] = "De-energization event"
psps["event_name"] = psps["event_name"].astype(str)
psps["location_text"] = psps["iou"].astype(str) + " | counties de-energized: " + psps["counties_de_energized"].astype(str)
psps["detail_text"] = "PSPS record from CPUC CA 2024 PSPS events."
psps["event_value"] = pd.to_numeric(psps["customers_de_energized"], errors="coerce")
psps["county_name"] = None
psps["plot_end_time"] = psps["end_time"]

# -------------------------------------------------------------------
# CPUC wildfire ignition events
# -------------------------------------------------------------------
wildfire = pd.read_excel(IGNITION_XLSX, sheet_name="Wildfire Events")
wildfire["start_time"] = pd.to_datetime(wildfire["Date"], errors="coerce")
wildfire = wildfire.dropna(subset=["start_time"]).copy()
wildfire = wildfire[(wildfire["start_time"].dt.year >= 2024) & (wildfire["start_time"].dt.year <= 2024)].copy()
wildfire["days_active"] = pd.to_numeric(wildfire["Days Active (until containment)"], errors="coerce")
wildfire["end_time"] = wildfire["start_time"] + pd.to_timedelta(wildfire["days_active"].fillna(0), unit="D")
wildfire.loc[wildfire["end_time"] <= wildfire["start_time"], "end_time"] = wildfire["start_time"] + pd.Timedelta(hours=12)
wildfire["source"] = "CPUC Ignition"
wildfire["event_type"] = "Wildfire ignition"
wildfire["event_name"] = wildfire["Wildfire Event"].astype(str)
wildfire["location_text"] = wildfire["Location"].astype(str) + " | " + wildfire["Entity with jurisdiction"].astype(str)
wildfire["detail_text"] = wildfire["Long cause of ignition"].astype(str)
wildfire["event_value"] = pd.to_numeric(wildfire["Size (acres)"], errors="coerce")
wildfire["county_name"] = wildfire.apply(
    lambda row: county_name_from_text(" ".join([str(row.get("Wildfire Event", "")), str(row.get("Location", "")), str(row.get("Entity with jurisdiction", ""))]), ca_counties["county_name"].tolist()),
    axis=1,
)
wildfire["plot_end_time"] = wildfire["end_time"]

# -------------------------------------------------------------------
# Participant interview references
# -------------------------------------------------------------------
interviews = pd.read_excel(PARTICIPANT_XLSX)
interviews["interview_time"] = pd.to_datetime(interviews["Interview Date"], errors="coerce")
interviews["start_time"] = interviews["Approx. Date of Event"].map(parse_any_datetime)
interviews = interviews.dropna(subset=["start_time"]).copy()
interviews = interviews[(interviews["start_time"].dt.year >= 2024) & (interviews["start_time"].dt.year <= 2024)].copy()
interviews["end_time"] = interviews["start_time"] + pd.Timedelta(hours=6)
interviews["source"] = "Interview reference"
interviews["event_type"] = interviews["Hazard Type"].astype(str)
interviews["event_name"] = interviews["Event Mentioned"].astype(str)
interviews["location_text"] = interviews.apply(
    lambda row: " | ".join([
        clean_text(row.get("Infrastructure Affected", "")),
        clean_text(row.get("Operational Impacts", "")),
        clean_text(row.get("Key Quote/Notes", "")),
    ]),
    axis=1,
)
interviews["detail_text"] = interviews["Cascading or Compounding"].astype(str)
interviews["event_value"] = np.nan
interviews["county_name"] = interviews.apply(
    lambda row: county_name_from_text(" ".join([str(row.get("Event Mentioned", "")), str(row.get("Infrastructure Affected", "")), str(row.get("Key Quote/Notes", ""))]), ca_counties["county_name"].tolist()),
    axis=1,
)
interviews["plot_end_time"] = interviews["end_time"]

# -------------------------------------------------------------------
# Combined timeline table
# -------------------------------------------------------------------
timeline_df = pd.concat([
    eaglei_agg[["source", "event_type", "event_name", "start_time", "end_time", "plot_end_time", "location_text", "county_name", "event_value", "detail_text"]],
    psps[["source", "event_type", "event_name", "start_time", "end_time", "plot_end_time", "location_text", "county_name", "event_value", "detail_text"]],
    wildfire[["source", "event_type", "event_name", "start_time", "end_time", "plot_end_time", "location_text", "county_name", "event_value", "detail_text"]],
    interviews[["source", "event_type", "event_name", "start_time", "end_time", "plot_end_time", "location_text", "county_name", "event_value", "detail_text"]],
], ignore_index=True, sort=False)

timeline_df = timeline_df.dropna(subset=["start_time"]).copy()
timeline_df["plot_end_time"] = pd.to_datetime(timeline_df["plot_end_time"], errors="coerce")
timeline_df.loc[timeline_df["plot_end_time"].isna() | (timeline_df["plot_end_time"] <= timeline_df["start_time"]), "plot_end_time"] = timeline_df["start_time"] + pd.Timedelta(hours=6)
timeline_df = timeline_df.sort_values(["source", "start_time", "event_name"]).reset_index(drop=True)

print(f"CA counties loaded: {len(ca_counties)}")
print(f"Eagle-I 2024 CA rows after aggregation: {len(eaglei_agg)}")
print(f"PSPS 2024 rows: {len(psps)}")
print(f"CPUC ignition 2024 rows: {len(wildfire)}")
print(f"Interview references with parseable 2024 dates: {len(interviews)}")
print(f"Combined timeline rows: {len(timeline_df)}")

In [ ]:
# Extreme-event sensitivity controls for Eagle-I county events
# Increase percentiles / floors to reduce flagged events.
OUTAGE_PERCENTILE = 0.95
SURGE_PERCENTILE = 0.95
MIN_COUNTY_OUTAGE_FOR_EXTREME = 300
MIN_SURGE_DELTA_FOR_EXTREME = 100

print("Eagle-I extreme-event tuning")
print(f"  OUTAGE_PERCENTILE: {OUTAGE_PERCENTILE:.2f}")
print(f"  SURGE_PERCENTILE: {SURGE_PERCENTILE:.2f}")
print(f"  MIN_COUNTY_OUTAGE_FOR_EXTREME: {MIN_COUNTY_OUTAGE_FOR_EXTREME}")
print(f"  MIN_SURGE_DELTA_FOR_EXTREME: {MIN_SURGE_DELTA_FOR_EXTREME}")

In [ ]:
# --- Pull tuning controls (with safe defaults if control cell not run) ---
outage_pct = float(globals().get("OUTAGE_PERCENTILE", 0.95))
surge_pct = float(globals().get("SURGE_PERCENTILE", 0.95))
min_outage_floor = float(globals().get("MIN_COUNTY_OUTAGE_FOR_EXTREME", 300))
min_surge_floor = float(globals().get("MIN_SURGE_DELTA_FOR_EXTREME", 100))

# --- Build Eagle-I statewide daily outage total timeline ---
eaglei_daily_county = (
    eaglei.assign(event_date=eaglei["start_time"].dt.floor("D"))
    .groupby(["event_date", "fips_code", "county"], as_index=False)
    .agg(customers_out=("customers_out", "max"))
)

statewide_daily = (
    eaglei_daily_county.groupby("event_date", as_index=False)["customers_out"]
    .sum()
    .rename(columns={"customers_out": "statewide_customers_out"})
)

# --- Identify county-level Eagle-I extreme outage events ---
county_q = (
    eaglei_daily_county.groupby("fips_code", as_index=False)["customers_out"]
    .quantile(outage_pct)
    .rename(columns={"customers_out": "county_outage_q"})
)

county_delta = eaglei_daily_county.sort_values(["fips_code", "event_date"]).copy()
county_delta["surge_delta"] = county_delta.groupby("fips_code")["customers_out"].diff().fillna(0)

delta_q = (
    county_delta[county_delta["surge_delta"] > 0]
    .groupby("fips_code", as_index=False)["surge_delta"]
    .quantile(surge_pct)
    .rename(columns={"surge_delta": "surge_q"})
)

county_extreme = (
    county_delta
    .merge(county_q, on="fips_code", how="left")
    .merge(delta_q, on="fips_code", how="left")
)
county_extreme["is_above_county_q"] = (
    (county_extreme["customers_out"] >= county_extreme["county_outage_q"].fillna(np.inf))
    & (county_extreme["customers_out"] >= min_outage_floor)
)
county_extreme["is_surge"] = (
    (county_extreme["surge_delta"] >= county_extreme["surge_q"].fillna(np.inf))
    & (county_extreme["surge_delta"] >= min_surge_floor)
)
county_extreme["is_eaglei_extreme"] = county_extreme["is_above_county_q"] | county_extreme["is_surge"]

eaglei_extreme_events = county_extreme[county_extreme["is_eaglei_extreme"]].copy()
eaglei_extreme_events["start_time"] = eaglei_extreme_events["event_date"]

# --- Build event bars from non-Eagle-I sources ---
event_bars = timeline_df[timeline_df["source"].isin(["PSPS", "CPUC Ignition", "Interview reference"])].copy()

lane_map = {
    "Interview reference": 1,
    "CPUC Ignition": 2,
    "PSPS": 3,
    "EAGLE-I extreme county": 4,
}
lane_color = {
    "Interview reference": SOURCE_COLORS["Interview reference"],
    "CPUC Ignition": SOURCE_COLORS["CPUC Ignition"],
    "PSPS": SOURCE_COLORS["PSPS"],
}

def make_span_trace(df, source_name):
    if df.empty:
        return None

    x, y, cd = [], [], []
    for _, row in df.iterrows():
        x.extend([row["start_time"], row["plot_end_time"], None])
        y.extend([lane_map[source_name], lane_map[source_name], None])
        payload = [
            clean_text(row.get("event_name", "")),
            clean_text(row.get("event_type", "")),
            clean_text(row.get("location_text", "")),
            clean_text(row.get("detail_text", "")),
            pd.to_datetime(row.get("start_time")).strftime("%Y-%m-%d %H:%M") if pd.notna(row.get("start_time")) else "",
            pd.to_datetime(row.get("end_time")).strftime("%Y-%m-%d %H:%M") if pd.notna(row.get("end_time")) else "",
        ]
        cd.extend([payload, payload, ["", "", "", "", "", ""]])

    return go.Scatter(
        x=x,
        y=y,
        mode="lines",
        line=dict(color=lane_color[source_name], width=11),
        opacity=0.85,
        name=source_name,
        yaxis="y2",
        customdata=cd,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Type: %{customdata[1]}<br>"
            "Location: %{customdata[2]}<br>"
            "Start: %{customdata[4]}<br>"
            "End: %{customdata[5]}<br>"
            "Details: %{customdata[3]}<extra></extra>"
        ),
    )

fig = go.Figure()

# Base: statewide Eagle-I outage totals
fig.add_trace(
    go.Bar(
        x=statewide_daily["event_date"],
        y=statewide_daily["statewide_customers_out"],
        name="EAGLE-I statewide outage total (daily)",
        marker_color="rgba(30, 90, 200, 0.45)",
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Statewide outages</b>: %{y:,.0f}<extra></extra>"
        ),
    )
)

# Overlay bars from other event files on secondary lane axis
for src in ["Interview reference", "CPUC Ignition", "PSPS"]:
    tr = make_span_trace(event_bars[event_bars["source"] == src], src)
    if tr is not None:
        fig.add_trace(tr)

# Add special county-level Eagle-I extreme markers
if not eaglei_extreme_events.empty:
    fig.add_trace(
        go.Scatter(
            x=eaglei_extreme_events["start_time"],
            y=np.full(len(eaglei_extreme_events), lane_map["EAGLE-I extreme county"]),
            mode="markers",
            name=f"EAGLE-I county extreme (q{int(outage_pct * 100)}/q{int(surge_pct * 100)})",
            yaxis="y2",
            marker=dict(size=8, symbol="diamond", color="crimson", line=dict(width=0.4, color="black")),
            customdata=np.column_stack([
                eaglei_extreme_events["county"].astype(str),
                eaglei_extreme_events["customers_out"].round(0),
                eaglei_extreme_events["county_outage_q"].round(0),
                eaglei_extreme_events["surge_delta"].round(0),
                eaglei_extreme_events["surge_q"].round(0),
                eaglei_extreme_events["is_above_county_q"].astype(str),
                eaglei_extreme_events["is_surge"].astype(str),
            ]),
            hovertemplate=(
                "<b>County</b>: %{customdata[0]}<br>"
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "Customers out: %{customdata[1]:,.0f}<br>"
                f"County outage q{int(outage_pct * 100)}: " + "%{customdata[2]:,.0f}<br>"
                "Surge delta: %{customdata[3]:,.0f}<br>"
                f"Surge q{int(surge_pct * 100)}: " + "%{customdata[4]:,.0f}<br>"
                "Above county outage threshold: %{customdata[5]}<br>"
                "Surge event: %{customdata[6]}<br>"
                f"Floors: outage >= {min_outage_floor:,.0f}, surge >= {min_surge_floor:,.0f}"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    template="plotly_white",
    height=760,
    title="California 2024 timeline: statewide Eagle-I outages + overlaid events",
    xaxis=dict(title="Time"),
    yaxis=dict(title="Statewide Eagle-I customers out (daily max by county)", rangemode="tozero"),
    yaxis2=dict(
        title="Event lanes",
        overlaying="y",
        side="right",
        range=[0.5, 4.5],
        tickmode="array",
        tickvals=[1, 2, 3, 4],
        ticktext=["Interview", "CPUC Ignition", "PSPS", "EAGLE-I county extreme"],
        showgrid=False,
    ),
    legend_title_text="Series",
    margin=dict(l=30, r=30, t=80, b=20),
)

fig.add_vline(x=pd.Timestamp("2024-01-01"), line_width=1, line_dash="dot", line_color="gray")
fig.show()

print(f"Statewide daily Eagle-I points: {len(statewide_daily)}")
print(f"Non-Eagle event bars: {len(event_bars)}")
print(f"Eagle-I county extreme events: {len(eaglei_extreme_events)}")
print(f"Using outage percentile={outage_pct:.2f}, surge percentile={surge_pct:.2f}, outage floor={min_outage_floor:.0f}, surge floor={min_surge_floor:.0f}")

eaglei_extreme_events[["event_date", "county", "customers_out", "county_outage_q", "surge_delta", "surge_q", "is_above_county_q", "is_surge"]].head(20)

In [ ]:
ca_counties_plot = ca_counties.to_crs(ANALYSIS_CRS).copy()
ca_outline = gpd.GeoSeries([ca_counties_plot.geometry.unary_union], crs=ANALYSIS_CRS)
transmission = load_transmission_lines()

# Spatialized layers
eaglei_map = eaglei_agg.merge(ca_counties_plot[["GEOID", "NAME", "geometry"]], left_on="fips_code", right_on="GEOID", how="left")
eaglei_map = gpd.GeoDataFrame(eaglei_map, geometry="geometry", crs=ANALYSIS_CRS)

wildfire_map = wildfire.copy()
wildfire_map["county_geom"] = wildfire_map["county_name"].str.lower().map(county_geom_lookup)
wildfire_map["county_centroid"] = wildfire_map["county_name"].str.lower().map(county_centroid_lookup)
wildfire_map["geometry"] = wildfire_map["county_geom"].where(wildfire_map["county_geom"].notna(), wildfire_map["county_centroid"])
wildfire_map["geometry"] = wildfire_map["geometry"].apply(lambda g: g if g is not None else Point(ca_centroid.x, ca_centroid.y))
wildfire_map = gpd.GeoDataFrame(wildfire_map, geometry="geometry", crs=ANALYSIS_CRS)

psps_map = psps.copy()
psps_map["geometry"] = [Point(ca_centroid.x + 0.08 * ((i % 4) - 1.5), ca_centroid.y + 0.06 * ((i % 3) - 1)) for i in range(len(psps_map))]
psps_map = gpd.GeoDataFrame(psps_map, geometry="geometry", crs=ANALYSIS_CRS)

interview_map = interviews.copy()
interview_map["county_centroid"] = interview_map["county_name"].str.lower().map(county_centroid_lookup)
interview_map["geometry"] = interview_map["county_centroid"].apply(lambda g: g if g is not None else None)
interview_map = gpd.GeoDataFrame(interview_map, geometry="geometry", crs=ANALYSIS_CRS)

map_start = pd.Timestamp("2024-01-01")
map_end = pd.Timestamp("2024-12-31")
FRAME_FREQ = "7D"
movie_days = pd.date_range(map_start, map_end, freq=FRAME_FREQ)
county_bounds = ca_counties_plot.total_bounds
county_vmax = float(np.nanmax(eaglei_map["customers_out"].fillna(0).values)) if len(eaglei_map) else 1.0
county_vmax = max(county_vmax, 1.0)

print(f"Counties: {len(ca_counties_plot)}")
print(f"Transmission features: {len(transmission)}")
print(f"Movie frames: {len(movie_days)} at {FRAME_FREQ}")
print(f"EAGLE-I map rows: {len(eaglei_map)}")
print(f"Wildfire map rows: {len(wildfire_map)}")
print(f"PSPS map rows: {len(psps_map)}")
print(f"Interview map rows: {len(interview_map)}")

In [ ]:
output_dir = Path("./outputs")
output_dir.mkdir(parents=True, exist_ok=True)
movie_path = output_dir / "ca_multihazard_2024.gif"

frames = []
for day in movie_days:
    day_start = pd.Timestamp(day).normalize()
    day_end = day_start + pd.Timedelta(days=1)

    fig, ax = plt.subplots(figsize=(15, 13))

    ca_counties_plot.boundary.plot(ax=ax, color="lightgray", linewidth=0.35, zorder=1)
    transmission.plot(ax=ax, color="black", linewidth=0.35, alpha=0.4, zorder=2)

    eagle_active = eaglei_map[(eaglei_map["start_time"] < day_end) & (eaglei_map["end_time"] >= day_start)]
    psps_active = psps_map[(psps_map["start_time"] < day_end) & (psps_map["end_time"] >= day_start)]
    wildfire_active = wildfire_map[(wildfire_map["start_time"] < day_end) & (wildfire_map["end_time"] >= day_start)]
    interview_active = interview_map[(interview_map["start_time"] < day_end) & (interview_map["end_time"] >= day_start)]

    if len(eagle_active) > 0:
        eagle_active.plot(
            ax=ax,
            column="customers_out",
            cmap="Blues",
            vmin=0,
            vmax=county_vmax,
            linewidth=0.15,
            edgecolor="navy",
            alpha=0.65,
            zorder=4,
        )

    wildfire_polys = wildfire_active[wildfire_active.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    wildfire_points = wildfire_active[wildfire_active.geometry.geom_type == "Point"]
    if len(wildfire_polys) > 0:
        wildfire_polys.plot(ax=ax, facecolor="tomato", edgecolor="darkred", linewidth=1.0, alpha=0.30, zorder=5)
    if len(wildfire_points) > 0:
        wildfire_points.plot(ax=ax, color="darkred", markersize=45, alpha=0.9, zorder=6)

    if len(psps_active) > 0:
        psps_active.plot(ax=ax, color="darkorange", markersize=55, marker="s", alpha=0.85, zorder=7)

    if len(interview_active) > 0:
        interview_active.plot(ax=ax, color="purple", markersize=45, marker="D", alpha=0.85, zorder=8)

    summary_text = (
        f"EAGLE-I active county rows: {len(eagle_active)}\n"
        f"PSPS active events: {len(psps_active)}\n"
        f"Wildfire events active: {len(wildfire_active)}\n"
        f"Interview references active: {len(interview_active)}"
    )
    ax.text(0.01, 0.99, summary_text, transform=ax.transAxes, va="top", ha="left", fontsize=10, bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))

    ax.set_xlim(county_bounds[0], county_bounds[2])
    ax.set_ylim(county_bounds[1], county_bounds[3])
    ax.set_title(f"California multi-hazard map on {day_start.strftime('%Y-%m-%d')}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.2, linestyle="--")

    legend_handles = [
        plt.Line2D([0], [0], color=SOURCE_COLORS["EAGLE-I"], lw=6, label="EAGLE-I county outages"),
        plt.Line2D([0], [0], color=SOURCE_COLORS["CPUC Ignition"], lw=6, label="CPUC wildfire ignition"),
        plt.Line2D([0], [0], color=SOURCE_COLORS["PSPS"], marker="s", linestyle="", markersize=10, label="PSPS event"),
        plt.Line2D([0], [0], color=SOURCE_COLORS["Interview reference"], marker="D", linestyle="", markersize=8, label="Interview reference"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=9, frameon=True)

    sm = ScalarMappable(norm=Normalize(vmin=0, vmax=county_vmax), cmap="Blues")
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.01)
    cbar.set_label("EAGLE-I customers out")

    plt.tight_layout()
    frames.append(geo_to_rgba(fig))
    plt.close(fig)

imageio.mimsave(movie_path, frames, duration=0.8)
print(f"Saved movie to {movie_path}")
movie_path

## Notes

- The timeline keeps only rows with parseable dates in 2024.
- The wildfire workbook currently has four 2024 ignition rows.
- The participant workbook may not yield any parseable 2024 event dates from the current sheet; those rows are kept in the raw source and can be adapted later if a better date parsing rule is needed.
- The map movie uses county polygons where the source provides a county or county-like location. When a source only gives a coarse or non-spatial description, the notebook falls back to a point marker near the California centroid so the event is still visible.

If you want, the next useful extension is to turn the event table into a single tidy parquet/CSV export so the timeline and movie can be re-rendered without rereading the source workbooks.